**Spam Message Detector**
A machine learning model that classifies text messages as Spam or Not Spam, using a dataset of 60 labeled examples (30 spam, 30 not spam) and a Multinomial Naive Bayes classifier.

## 1. Load and Explore the Data

In [ ]:
import pandas as pd
import sklearn

df = pd.read_csv("message.csv", encoding="latin-1")
df.head(10)

,Source,Message,Label
0,2438001000001,"Congratulations! You have won N1,000,000. Clai...",Spam
1,2438001000002,You have been selected for a FREE cash reward....,Spam
2,2438001000003,WIN BIG TODAY! Send your details to receive yo...,Spam
3,2438001000004,Your account has won a special bonus. Claim be...,Spam
4,2438001000005,"Congratulations, lucky winner! You have won a ...",Spam
5,2438001000006,"Get FREE airtime worth N10,000 today. Click he...",Spam
6,2438001000007,Urgent! Your reward is waiting. Confirm your d...,Spam
7,2438001000008,You have won our weekly cash giveaway. Reply Y...,Spam
8,2438001000009,Exclusive offer! Get a FREE shopping voucher t...,Spam
9,2438001000010,Congratulations! Your number was chosen for a ...,Spam


In [ ]:
print(df.columns)
print(df.shape)

Index(['Source', 'Message', 'Label'], dtype='object')
(60, 3)


## 2. Preprocessing
We drop the 'Source' column since a phone number has no bearing on whether a message is spam, and convert the 'Label' column from text ("Spam"/"Not Spam") to numbers (1/10) since the model requires numeric input.

In [ ]:
df = df[['Message', 'Label']]
df['Label'] = df['Label'].map({'Spam': 1, 'Not Spam': 0})
print(df['Label'].value_counts())
print(df.isnull().sum())

Label
1    30
0    30
Name: count, dtype: int64
Message    0
Label      0
dtype: int64


## 3. Train/Test Split
We split the data 80/20, using stratification to keep the spam/not-spam ratio consistent in both sets. This is important given the small dataset size.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['Message'], df['Label'], test_size=0.2, random_state=42, stratify=df['Label']
)
print(X_train.shape, X_test.shape)

(48,) (12,)


## 4. Feature Extraction (Vectorization)
Text can't be fed directly into a model, so we convert each message into a vector of word counts using 'CountVectorizer'. English stop words (e.g "the", "is") are removed since they carry no useful signal for spam detection. The vectorizer is fit only on the training data to avoid leaking test information into the model.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(stop_words='english', lowercase=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(X_train_vec.shape)

(48, 171)


## 5. Train the model
We use Multinomial Naive Bayes, a standard, lightweight algorithm for text classification that works well even on small datasets.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_vec, y_train)

MultinomialNB()

## 6. Evaluate the Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

preds = model.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))
print(confusion_matrix(y_test, preds))

Accuracy: 0.9166666666666666
              precision    recall  f1-score   support

           0       1.00      0.83      0.91         6
           1       0.86      1.00      0.92         6

    accuracy                           0.92        12
   macro avg       0.93      0.92      0.92        12
weighted avg       0.93      0.92      0.92        12

[[5 1]
 [0 6]]


The model reached ~92% accuracy on the test set. Or more precisely, an accuracy of 91.6%. The confusion matrix shows it caught every spam message (recall = 1.00 for class 1), with one non-spam message incorrectly flagged as spam. This is a reasonable trade-off for a spam filter — missing spam is generally worse than occasionally over-flagging a legitimate message. With only 12 test examples, this exact percentage should be read as a rough indicator, not a precise measure — a different train/test split could shift it noticeably.

## 7. Testing on New, Unseen Messages

In [ ]:
sample = ["Congratulations! You have been selected to win a free vacation, click now!"]
sample_vec = vectorizer.transform(sample)
print("Prediction:", model.predict(sample_vec))
print("Probabilities.", model.predict_proba(sample_vec))

Prediction: [1]
Probabilities. [[0.00988858 0.99011142]]


In [ ]:
sample = ["Congrats on winning the game last night."]
sample_vec = vectorizer.transform(sample)
print("Prediction:", model.predict(sample_vec))
print("Probabilities.", model.predict_proba(sample_vec))

Prediction: [0]
Probabilities. [[0.5 0.5]]


The first message contains strong spam indicators ("won", "free", "click") and was classified as spam with 99% confidence. The second message shares some vocabulary with spam ("congrats", "winning") but lacks other spam signals, and the model was split almost exactly 50/50 — correctly reflecting that this message is genuinely ambiguous rather than being confidently wrong.

## 8. Conclusion and Limitations
This project demonstrates a full text classification pipeline: data cleaning, vectorization, model training, and evaluation. The model performs well on clear-cut cases but shows realistic uncertainty on borderline messages.
Key limitations: the dataset is very small (60 messages), so performance metrics carry high variance and shouldn't be over-interpreted. Future improvements could include a larger dataset, TF-IDF vectorization instead of raw counts, or testing alternative models (e.g. Logistic Regression, SVM).